In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
files.upload()  # Selecione o arquivo kaggle.json

In [ ]:
import os
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

In [ ]:
!pip install kaggle -q
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip intel-image-classification.zip -d /content/intel-data/ -q
print("Download concluído!")

In [ ]:
!unzip -o /content/intel-image-classification.zip -d /content/intel-data/

In [ ]:
import os

dataset_path = '/content/intel-data/seg_train/seg_train'
classes = os.listdir(dataset_path)
print(f"Classes encontradas: {classes}")

for cls in sorted(classes):
    n = len(os.listdir(os.path.join(dataset_path, cls)))
    print(f"  {cls}: {n} imagens")

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow versão: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Visualizar uma amostra de cada classe
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, cls in enumerate(sorted(classes)):
    cls_path = os.path.join(dataset_path, cls)
    img_file = os.listdir(cls_path)[0]
    img = mpimg.imread(os.path.join(cls_path, img_file))
    axes[idx].imshow(img)
    axes[idx].set_title(f'Classe: {cls}', fontsize=13, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Amostras por Classe — Intel Image Classification', fontsize=16)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/transfer_learning_samples.png', dpi=150)
plt.show()

In [ ]:
# Distribuição por classe
train_counts = {cls: len(os.listdir(os.path.join(dataset_path, cls))) for cls in sorted(classes)}

plt.figure(figsize=(10, 5))
plt.bar(train_counts.keys(), train_counts.values(), color='steelblue', edgecolor='black')
plt.title('Distribuição de Imagens por Classe (Treino)', fontsize=14)
plt.xlabel('Classe')
plt.ylabel('Número de imagens')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
NUM_CLASSES = 6

TRAIN_DIR = '/content/intel-data/seg_train/seg_train'
TEST_DIR  = '/content/intel-data/seg_test/seg_test'

In [ ]:
# Augmentation para treino
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Apenas normalização para teste
test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f"Classes mapeadas: {train_generator.class_indices}")

In [ ]:
def build_baseline_model(num_classes):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(*IMG_SIZE, 3)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

baseline_model = build_baseline_model(NUM_CLASSES)
baseline_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
baseline_model.summary()

In [ ]:
callbacks_baseline = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history_baseline = baseline_model.fit(
    train_generator, epochs=20,
    validation_data=val_generator,
    callbacks=callbacks_baseline, verbose=1
)

In [ ]:
# Carregar VGG16 sem a cabeça classificadora original
base_model = VGG16(
    weights='imagenet',
    include_top=False,       # Remove a camada softmax de 1000 classes
    input_shape=(*IMG_SIZE, 3)
)

# Congelar TODOS os pesos do VGG16
base_model.trainable = False
print(f"Parâmetros totais do VGG16: {base_model.count_params():,}")

In [ ]:
# Nova cabeça classificadora para 6 classes
def build_transfer_model(base, num_classes):
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

transfer_model = build_transfer_model(base_model, NUM_CLASSES)
transfer_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
transfer_model.summary()

In [ ]:
callbacks_transfer = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/content/drive/MyDrive/best_transfer_model.h5', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print("Fase 1: Treinando apenas a nova camada de classificação...")
history_transfer = transfer_model.fit(
    train_generator, epochs=20,
    validation_data=val_generator,
    callbacks=callbacks_transfer, verbose=1
)

In [ ]:
# Descongelar as últimas 4 camadas do VGG16
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 4

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Camadas treináveis: {sum(1 for l in base_model.layers if l.trainable)}")

In [ ]:
# Recompilar com learning rate 10x menor
transfer_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Fase 2: Fine-tuning das últimas camadas do VGG16...")
history_finetune = transfer_model.fit(
    train_generator, epochs=10,
    validation_data=val_generator,
    callbacks=callbacks_transfer, verbose=1
)

In [ ]:
def plot_history(histories, labels, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    colors = ['steelblue', 'tomato']

    for hist, label, color in zip(histories, labels, colors):
        if isinstance(hist, list):
            acc     = sum([h.history['accuracy'] for h in hist], [])
            val_acc = sum([h.history['val_accuracy'] for h in hist], [])
            loss    = sum([h.history['loss'] for h in hist], [])
            val_loss= sum([h.history['val_loss'] for h in hist], [])
        else:
            acc, val_acc = hist.history['accuracy'], hist.history['val_accuracy']
            loss, val_loss = hist.history['loss'], hist.history['val_loss']

        epochs = range(1, len(acc) + 1)
        ax1.plot(epochs, acc,     '-',  color=color, label=f'{label} (treino)', linewidth=2)
        ax1.plot(epochs, val_acc, '--', color=color, label=f'{label} (val)',    linewidth=2)
        ax2.plot(epochs, loss,     '-',  color=color, label=f'{label} (treino)', linewidth=2)
        ax2.plot(epochs, val_loss, '--', color=color, label=f'{label} (val)',    linewidth=2)

    for ax, ylabel, title_ax in zip([ax1, ax2], ['Acurácia', 'Loss'], ['Acurácia', 'Loss']):
        ax.set_title(f'{title_ax} ao longo do treino', fontsize=13)
        ax.set_xlabel('Épocas')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/learning_curves.png', dpi=150)
    plt.show()

plot_history(
    histories=[history_baseline, [history_transfer, history_finetune]],
    labels=['Baseline (CNN do zero)', 'Transfer Learning + Fine-Tuning'],
    title='Comparação: CNN do Zero vs Transfer Learning'
)

In [ ]:
def evaluate_model(model, generator, name):
    generator.reset()
    loss, acc = model.evaluate(generator, verbose=0)
    print(f"\n{'='*50}\nModelo: {name}\n  Loss: {loss:.4f} | Acurácia: {acc*100:.2f}%\n{'='*50}")
    return loss, acc

loss_b, acc_b = evaluate_model(baseline_model,  test_generator, "Baseline CNN do Zero")
loss_t, acc_t = evaluate_model(transfer_model,  test_generator, "Transfer Learning + Fine-Tuning")

In [ ]:
test_generator.reset()
preds  = transfer_model.predict(test_generator, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de Confusão — Transfer Learning + Fine-Tuning', fontsize=14)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
def show_predictions(model, generator, class_names, n=12):
    generator.reset()
    images, labels = next(generator)
    preds = model.predict(images, verbose=0)

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(images[i])
        true_cls = class_names[np.argmax(labels[i])]
        pred_cls = class_names[np.argmax(preds[i])]
        conf     = np.max(preds[i]) * 100
        color    = 'green' if true_cls == pred_cls else 'red'
        ax.set_title(f'Real: {true_cls}\nPred: {pred_cls} ({conf:.1f}%)', color=color, fontsize=9)
        ax.axis('off')

    plt.suptitle('Verde = Correto | Vermelho = Errado', fontsize=13)
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/prediction_samples.png', dpi=150)
    plt.show()

show_predictions(transfer_model, test_generator, class_names)

In [ ]:
transfer_model.save('/content/drive/MyDrive/transfer_learning_intel_final.h5')

print("\n" + "="*60)
print("  RESUMO FINAL")
print("="*60)
print(f"  Baseline CNN do Zero:            {acc_b*100:.2f}%")
print(f"  Transfer Learning + Fine-Tuning: {acc_t*100:.2f}%")
print(f"  Ganho com Transfer Learning:     +{(acc_t - acc_b)*100:.2f}%")
print("="*60)